## Caution
Before procceding with training, please make sure that you manually reviewed and labeled is_safe collumn on generated csv files from running gui.py

In [6]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from pathlib import Path

## Data Preparation
Combines all labeled logs into one then proccessed

In [7]:
LABELED_CSV_DIR = "../data/labeled_logs"
dfs = [pd.read_csv(f) for f in Path(LABELED_CSV_DIR).glob("*.csv")]
df = pd.concat(dfs, ignore_index=True)

df = df[df["dist"].notna()]
df = df[df["light_state"] != "unknown"]

# Encode light state
le = LabelEncoder()
df["light_enc"] = le.fit_transform(df["light_state"])  # N/A=0, green=1, red=2, yellow=3

# Encode is_oncoming (True/False -> 1/0)
df["is_oncoming"] = df["is_oncoming"].astype(int)

features = ["class", "dist", "delta_dist", "light_enc", "is_oncoming"]
X = df[features]
y = df["is_safe"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Class distribution:\n{y.value_counts()}")


Train size: 11924, Test size: 2981
Class distribution:
is_safe
0    9434
1    5471
Name: count, dtype: int64


## Classifiers

In [8]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision Tree":       DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
}

### Optional: Print classifier classificaiton reports

In [9]:
trained = {}

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    trained[name] = clf
    print(f"\n{'='*50}")
    print(f" {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, clf.predict(X_test)))


 Logistic Regression
              precision    recall  f1-score   support

           0       0.89      0.62      0.73      1903
           1       0.56      0.86      0.68      1078

    accuracy                           0.71      2981
   macro avg       0.72      0.74      0.70      2981
weighted avg       0.77      0.71      0.71      2981


 Decision Tree
              precision    recall  f1-score   support

           0       0.90      0.79      0.84      1903
           1       0.70      0.85      0.77      1078

    accuracy                           0.81      2981
   macro avg       0.80      0.82      0.81      2981
weighted avg       0.83      0.81      0.82      2981


 Random Forest
              precision    recall  f1-score   support

           0       0.90      0.79      0.84      1903
           1       0.70      0.85      0.77      1078

    accuracy                           0.81      2981
   macro avg       0.80      0.82      0.81      2981
weighted avg       0

## Save trained classifiers models
! IMPORTANT !

In [10]:
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for name, clf in trained.items():
    filename = name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(clf, MODEL_DIR / filename)
    print(f"Saved: {MODEL_DIR / filename}")

# Decode light_state (same for all models)
joblib.dump(le, MODEL_DIR / "label_encoder.pkl")
print(f"Saved: {MODEL_DIR / 'label_encoder.pkl'}")

Saved: ../models/logistic_regression.pkl
Saved: ../models/decision_tree.pkl
Saved: ../models/random_forest.pkl
Saved: ../models/label_encoder.pkl
